# 02 — SOH 정의

**무엇을 확인하는가**

1. 상위 코드의 SOH 정의를 그대로 계산해 본다
2. **SOC span 나눗셈을 뺀 변형과 나란히 본다** (`LAB-005`)
3. 폐기 임계 0.825 밴드에 몇 개가 걸리는지 센다 (`LAB-001`)
4. SOH > 1 인 셀을 찾는다 (Zn-ion 에서 보고된 현상)

정의는 이것입니다.

```
soh = max(cycle["discharge_capacity_in_Ah"]) / nominal_capacity / SOC_span
```

`SOC_span` 항의 타당성이 미해결입니다. **끄는 것이 옳다고 주장하는 것이
아니라, 갈리는지 보려는 것입니다.**

## 0. 부트스트랩

In [ ]:
import sys
from pathlib import Path

# 노트북에서 저장소 루트를 import 경로에 넣습니다.
REPO = Path.cwd()
while not (REPO / "run.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("저장소 루트:", REPO)

import pickle

import numpy as np

from verify import load_config
from verify import soh as soh_mod
from verify import labels as labels_mod

config = load_config()
EXTRACT = Path(config["EXTRACT_DIR"])

## 1. 셀 하나로 정의를 눈으로 확인

`nominal` 이 pkl 값인지 덮어쓴 값인지, `SOC_span` 이 얼마인지 먼저 보십시오.

In [ ]:
SUBSET = "SNL"      # nominal 덮어쓰기 사례가 있는 서브셋
directory = EXTRACT / SUBSET
files = sorted(f for f in directory.iterdir() if f.suffix == ".pkl")
print(f"{SUBSET}: {len(files)} cells")

name = files[0].name
with open(files[0], "rb") as f:
    data = pickle.load(f)

nominal = soh_mod.nominal_capacity(name, data)
span = soh_mod.soc_span_main(data)
print(f"\n{name}")
print(f"  pkl nominal      {data['nominal_capacity_in_Ah']}")
print(f"  적용 nominal     {nominal}   (덮어쓰기 여부에 주의)")
print(f"  SOC_interval     {data['SOC_interval']}  → span {span}")
print(f"  마지막 SOH       {soh_mod.last_cycle_soh(data, name):.4f}")
print(f"  사이클 수        {len(data['cycle_data'])}")

## 2. 두 변형을 나란히

`use_soc_span=True` 가 상위 코드와 같습니다.

In [ ]:
numbers, with_span = soh_mod.soh_curve(data, name, use_soc_span=True)
_, without_span = soh_mod.soh_curve(data, name, use_soc_span=False)

print(f"{'cycle':>8}  {'span 적용':>10}  {'span 뺌':>10}")
for i in list(range(0, len(numbers), max(1, len(numbers) // 10)))[:10]:
    print(f"{numbers[i]:8.0f}  {with_span[i]:10.4f}  {without_span[i]:10.4f}")

print()
print(f"span 적용 최종 {with_span[-1]:.4f} / span 뺌 최종 {without_span[-1]:.4f}")
print(f"두 값의 비: {with_span[-1] / without_span[-1]:.6f}  (= 1/span)")

## 3. 서브셋별 — 0.825 밴드와 SOH > 1

`last_cycle_soh` 는 곡선의 최솟값이 아니라 **마지막 사이클** 입니다
(`Extract_life_labels.py:110`). 용량이 되살아난 셀은 곡선이 0.8 아래로
내려갔더라도 여기서 폐기됩니다.

In [ ]:
ROWS = []
for path in sorted(p for p in EXTRACT.iterdir() if p.is_dir()):
    if path.name in ("Life labels", "READMEs"):
        continue
    route = labels_mod.route_of(path.name)
    if route in (labels_mod.ROUTE_FARASIS, labels_mod.ROUTE_CALB):
        print(f"  {path.name:14} 재현불가 경로 — 건너뜁니다 ({route})")
        continue

    files = sorted(f for f in path.iterdir() if f.suffix == ".pkl")
    abandoned = band = over_one = 0
    for file in files:
        with open(file, "rb") as f:
            cell = pickle.load(f)
        value = soh_mod.last_cycle_soh(cell, file.name)
        if value >= 0.825:
            abandoned += 1
        elif value > 0.8:
            band += 1
        _, curve = soh_mod.soh_curve(cell, file.name)
        if np.nanmax(curve) > 1.0:
            over_one += 1

    ROWS.append({"subset": path.name, "cells": len(files),
                 "abandoned": abandoned, "band": band, "soh_over_1": over_one})
    print(f"  {path.name:14} {len(files):4} cells   폐기 {abandoned:3}   "
          f"외삽밴드 {band:3}   SOH>1 {over_one:3}")

## 4. 무엇이 보이는가

- **폐기** 는 라벨이 아예 만들어지지 않는 셀입니다. 배포 라벨 파일에 없어야
  맞습니다. 있으면 그것이 불일치입니다 (03 에서 확인).
- **외삽밴드** 는 (0.8, 0.825) 구간에서 끝난 셀입니다. 라벨이 관측이 아니라
  회귀 예측값입니다.
- **SOH > 1** 은 공칭용량보다 방전용량이 큰 셀입니다. SOC span 나눗셈이나
  공칭용량 정의 중 하나가 그 셀에 안 맞는다는 신호입니다.

여기서 나온 숫자를 03 의 도메인 롤업과 맞춰 보십시오. 어긋나면 어느 쪽이
틀렸는지가 아니라 **정의가 두 군데서 다르게 쓰였는지** 를 먼저 의심하십시오.